<div align="center">

<h2>ESCUELA POLITÉCNICA NACIONAL</h2>

<h3>FACULTAD DE INGENIERÍA DE SISTEMAS</h3>

<p><strong>Materia:</strong> Recuperación de la Información</p>

<p><strong>Estudiante:</strong> Estéfano Condoy</p>

</div>

<hr>

# 04 BM25

## Objetivos

* Implementar BM25 y comparar su ranking con el de TF-IDF + coseno sobre la misma colección de documentos.
* Calcular el aporte de cada término en BM25, identificar qué controlan $k_1$ y $b$, y justificar un cambio de posición usando las estadísticas de los documentos.

**1. Carga el corpus de Project Gutenberg**

Utilice el corpus de Project Gutenberg y el preprocesamiento de la práctica anterior.

Recupere una tabla con los campos **Identificador de documento | Título | Texto**

Calcule la columna `tokens` para la tabla, con lista de tokens normalizados de cada documento.

Calcula DF e IDF.

In [7]:
# Aquí importo las herramientas para preparar el corpus y mostrar las tablas.
import re
from collections import Counter
from math import log, log1p

import nltk
import numpy as np
import pandas as pd
from IPython.display import display
from nltk.corpus import gutenberg

# Aquí cargo los 18 documentos de Gutenberg que usé en el deber anterior.
try:
    ids_documentos = list(map(str, gutenberg.fileids()))
except LookupError:
    nltk.download("gutenberg", raise_on_error=True)
    ids_documentos = list(map(str, gutenberg.fileids()))

# Aquí aplico el mismo preprocesamiento: minúsculas y apóstrofos internos.
TOKEN_RE = re.compile(r"[^\W_]+(?:'[^\W_]+)?", flags=re.UNICODE)

def tokenizar(texto):
    return TOKEN_RE.findall(texto.lower().replace("\u2019", "'"))

# Aquí guardo el ID, el título del encabezado y el texto original de cada libro.
filas_corpus = []
for doc_id in ids_documentos:
    texto = gutenberg.raw(doc_id)
    encabezado = texto.lstrip().splitlines()[0].strip().strip("[]")
    titulo = encabezado.split(" by ", 1)[0].rstrip(",")
    filas_corpus.append({"doc_id": doc_id, "Título": titulo, "Texto": texto})

corpus = pd.DataFrame(filas_corpus)
corpus["tokens"] = corpus["Texto"].map(tokenizar)
titulos = dict(zip(corpus["doc_id"], corpus["Título"]))
indice_documento = {doc_id: i for i, doc_id in enumerate(ids_documentos)}
total_documentos = len(corpus)

# Aquí calculo las longitudes en tokens y la longitud promedio de la colección.
longitudes = corpus["tokens"].map(len).to_numpy()
longitud_media = longitudes.mean()

# Aquí cuento TF por documento y sumo uno al DF por cada documento con el término.
tf_documentos = [Counter(tokens) for tokens in corpus["tokens"]]
df = Counter()
for conteos in tf_documentos:
    df.update(conteos.keys())

# Aquí calculo los dos IDF una sola vez con el DF de todo el corpus.
idf_tfidf = {termino: log(total_documentos / frecuencia) + 1
             for termino, frecuencia in df.items()}
# Aquí uso para BM25 el IDF con log(1 + x) documentado por Apache Lucene.
# https://lucene.apache.org/core/10_5_2/core/org/apache/lucene/search/similarities/BM25Similarity.html
idf_bm25 = {termino: log1p((total_documentos - frecuencia + 0.5) / (frecuencia + 0.5))
            for termino, frecuencia in df.items()}

# Aquí muestro la tabla y un ejemplo de los DF e IDF calculados.
with pd.option_context("display.max_colwidth", 55):
    display(corpus)
print(f"Documentos: {total_documentos} | Vocabulario: {len(df):,} términos")
print(f"Longitud media del corpus: {longitud_media:,.2f} tokens")
display(pd.DataFrame([
    {"Término": t, "DF": df[t], "IDF TF-IDF": idf_tfidf[t], "IDF BM25": idf_bm25[t]}
    for t in ("time", "whale", "queequeg")
]).round(6))

,doc_id,Título,Texto,tokens
0,austen-emma.txt,Emma,[Emma by Jane Austen 1816]\n\nVOLUME I\n\nCHAPTER I...,"[emma, by, jane, austen, 1816, volume, i, chapter, ..."
1,austen-persuasion.txt,Persuasion,[Persuasion by Jane Austen 1818]\n\n\nChapter 1\n\n...,"[persuasion, by, jane, austen, 1818, chapter, 1, si..."
2,austen-sense.txt,Sense and Sensibility,[Sense and Sensibility by Jane Austen 1811]\n\nCHAP...,"[sense, and, sensibility, by, jane, austen, 1811, c..."
3,bible-kjv.txt,The King James Bible,[The King James Bible]\n\nThe Old Testament of the ...,"[the, king, james, bible, the, old, testament, of, ..."
4,blake-poems.txt,Poems,[Poems by William Blake 1789]\n\n \nSONGS OF INNOCE...,"[poems, by, william, blake, 1789, songs, of, innoce..."
5,bryant-stories.txt,Stories to Tell to Children,[Stories to Tell to Children by Sara Cone Bryant 19...,"[stories, to, tell, to, children, by, sara, cone, b..."
6,burgess-busterbrown.txt,The Adventures of Buster Bear,[The Adventures of Buster Bear by Thornton W. Burge...,"[the, adventures, of, buster, bear, by, thornton, w..."
7,carroll-alice.txt,Alice's Adventures in Wonderland,[Alice's Adventures in Wonderland by Lewis Carroll ...,"[alice's, adventures, in, wonderland, by, lewis, ca..."
8,chesterton-ball.txt,The Ball and The Cross,[The Ball and The Cross by G.K. Chesterton 1909]\n\...,"[the, ball, and, the, cross, by, g, k, chesterton, ..."
9,chesterton-brown.txt,The Wisdom of Father Brown,[The Wisdom of Father Brown by G. K. Chesterton 191...,"[the, wisdom, of, father, brown, by, g, k, chestert..."


Documentos: 18 | Vocabulario: 43,825 términos
Longitud media del corpus: 121,225.78 tokens


,Término,DF,IDF TF-IDF,IDF BM25
0,time,18,1.000000,0.026668
1,whale,6,2.098612,1.072637
2,queequeg,1,3.890372,2.538974


**2. Benchmarking**

Seleccione **ocho consultas en el idioma del corpus** y registra en una línea qué busca el usuario con cada una. Incluye términos muy frecuentes, términos poco frecuentes, y consultas de distintas longitudes. Usa $df$ para justificar las elecciones.

Para **TF-IDF + coseno** conserva la representación dispersa de la matriz.

Para **BM25** define $k_1=1.2$, $b=0.75$. Mantén intactas las estadísticas de la colección al cambiar de consulta.

Para cada consulta, ordena por puntuación descendente y desempata por `doc_id` ascendente (trata los IDs como cadenas en ambos métodos). Presenta hasta diez documentos con score positivo por método. Una consulta sin términos en el vocabulario produce dos listas vacías.

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Aquí escojo ocho consultas en inglés y escribo qué busca el usuario.
consultas = {
    "time": ("Pasajes que hablen del tiempo.",
             "Un término presente en los 18 documentos."),
    "whale ship captain": ("Relatos sobre ballenas y capitanes de barco.",
                            "Tres términos con distinto alcance. whale tiene el menor DF."),
    "marriage family fortune": ("Novelas sobre matrimonio y patrimonio familiar.",
                                "Tres términos con DF similares, entre 12 y 13."),
    "king crown battle": ("Historias de reyes, coronas y batallas.",
                          "king y crown son más frecuentes en el corpus que battle."),
    "faith prayer god heaven angels": ("Pasajes sobre fe, oración y el cielo.",
                                       "Cinco términos con DF entre 10 y 16."),
    "queequeg": ("Pasajes donde aparece el personaje Queequeg.",
                 "Un nombre propio que solo aparece en un documento."),
    "cheshire gryphon": ("Pasajes sobre el gato de Cheshire o el grifo de Alicia.",
                          "Dos términos raros con DF de 3 y 2."),
    "thermonuclear microprocessor": ("Textos sobre tecnología termonuclear o microprocesadores.",
                                     "Caso de control con DF cero para ambos términos."),
}
tokens_consultas = {consulta: tokenizar(consulta) for consulta in consultas}
display(pd.DataFrame([
    {
        "Consulta": consulta,
        "Qué busca el usuario": intencion,
        "Longitud": len(tokens_consultas[consulta]),
        "DF por término": ", ".join(f"{t}={df[t]}" for t in tokens_consultas[consulta]),
        "Justificación por DF": motivo,
    }
    for consulta, (intencion, motivo) in consultas.items()
]))

# Aquí ajusto TF-IDF con el mismo tokenizador y conservo la matriz dispersa.
vectorizador = TfidfVectorizer(
    analyzer=tokenizar, use_idf=True, smooth_idf=False, sublinear_tf=False, norm=None
)
tfidf_documentos = vectorizador.fit_transform(corpus["Texto"]).tocsr()
vocabulario = vectorizador.get_feature_names_out()
np.testing.assert_allclose(vectorizador.idf_, [idf_tfidf[t] for t in vocabulario])

# Aquí transformo las consultas con el vocabulario y los pesos del corpus.
tfidf_consultas = vectorizador.transform(list(consultas))
cosenos = cosine_similarity(
    tfidf_consultas, tfidf_documentos, dense_output=False
).tocsr()

# Aquí calculo el aporte de cada término distinto y sumo los aportes por documento.
K1_BASE, B_BASE = 1.2, 0.75

def calcular_bm25(tokens, k1=K1_BASE, b=B_BASE):
    # Aquí b pondera la longitud relativa y k1 controla la saturación de TF.
    ajuste_longitud = k1 * (1 - b + b * longitudes / longitud_media)
    aportes = {}
    for termino in dict.fromkeys(tokens):
        if termino not in idf_bm25:
            continue
        tf = np.array([conteos[termino] for conteos in tf_documentos])
        aportes[termino] = idf_bm25[termino] * (k1 + 1) * tf / (tf + ajuste_longitud)
    puntuaciones = sum(aportes.values(), np.zeros(total_documentos))
    return puntuaciones, aportes

# Aquí descarto scores cero y desempato por el ID de texto en orden ascendente.
def ordenar_positivos(pares):
    return sorted(
        [(doc_id, float(score)) for doc_id, score in pares if score > 0],
        key=lambda par: (-par[1], par[0]),
    )

# Aquí extraigo únicamente los cosenos almacenados y preparo ambos top 10.
rankings_tfidf, rankings_bm25, aportes_base = {}, {}, {}
for i, (consulta, tokens) in enumerate(tokens_consultas.items()):
    fila_coseno = cosenos[i:i + 1]
    pares_coseno = zip(
        (ids_documentos[j] for j in fila_coseno.indices), fila_coseno.data
    )
    scores_bm25, aportes_base[consulta] = calcular_bm25(tokens)
    rankings_tfidf[consulta] = ordenar_positivos(pares_coseno)[:10]
    rankings_bm25[consulta] = ordenar_positivos(zip(ids_documentos, scores_bm25))[:10]

,Consulta,Qué busca el usuario,Longitud,DF por término,Justificación por DF
0,time,Pasajes que hablen del tiempo.,1,time=18,Un término presente en los 18 documentos.
1,whale ship captain,Relatos sobre ballenas y capitanes de barco.,3,"whale=6, ship=13, captain=9",Tres términos con distinto alcance. whale tien...
2,marriage family fortune,Novelas sobre matrimonio y patrimonio familiar.,3,"marriage=12, family=13, fortune=12","Tres términos con DF similares, entre 12 y 13."
3,king crown battle,"Historias de reyes, coronas y batallas.",3,"king=15, crown=15, battle=9",king y crown son más frecuentes en el corpus q...
4,faith prayer god heaven angels,"Pasajes sobre fe, oración y el cielo.",5,"faith=15, prayer=10, god=16, heaven=13, angels=11",Cinco términos con DF entre 10 y 16.
5,queequeg,Pasajes donde aparece el personaje Queequeg.,1,queequeg=1,Un nombre propio que solo aparece en un docume...
6,cheshire gryphon,Pasajes sobre el gato de Cheshire o el grifo d...,2,"cheshire=3, gryphon=2",Dos términos raros con DF de 3 y 2.
7,thermonuclear microprocessor,Textos sobre tecnología termonuclear o micropr...,2,"thermonuclear=0, microprocessor=0",Caso de control con DF cero para ambos términos.


**3. Compara los rankings**

Muestra por consulta una tabla como esta. Usa los IDs para comparar.

| Posición | ID y título TF-IDF | Coseno | ID y título BM25 | Score BM25 |
|---:|---|---:|---|---:|
| 1 | … | … | … | … |

Calcula cuántos IDs comparten los dos top 10 con $|T_{\mathrm{TFIDF}}\cap T_{\mathrm{BM25}}|$. Reporta también el tamaño real de cada lista y el primer documento de cada método. Dos listas pueden tener muchos IDs en común y ordenarlos de forma distinta.

Elige una consulta y dos documentos que cambien de posición o cuya distancia en el ranking cambie. Registra $|d|$, $f(t,d)$, $df(t)$ y el aporte BM25 de cada término de consulta para ambos documentos. Relaciona esas cifras con la saturación, la longitud o la rareza del término. Usa los scores de cada método para explicar su propio ranking.

**4. Identifica el efecto de los parámetros**

Mantén fijos la colección y el preprocesamiento. Escoge una consulta con coincidencias en documentos de distintas longitudes y frecuencias.

| Ejecución | $k_1$ | $b$ | Qué examinar |
|---|---:|---:|---|
| Base | 1.2 | 0.75 | Posiciones y aportes de referencia |
| Sin ajuste por longitud | 1.2 | 0.00 | Cambios en documentos más cortos o más largos que la media |
| Mayor efecto de las repeticiones | 2.0 | 0.75 | Cambios en el aporte de términos con frecuencias distintas |

Muestra las posiciones de los mismos documentos en las tres configuraciones. Si no se mueven, compara sus aportes por término y explica por qué el cambio de puntuación no alteró su orden.